In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

ROOT = Path.home() / "Desktop" / "FinCascade"

DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"

MODEL_DIR.mkdir(exist_ok=True)

print("Systemic Stress ML environment ready.")

Systemic Stress ML environment ready.


In [2]:
returns = pd.read_csv(
    DATA_DIR / "market_returns_model.csv",
    index_col=0,
    parse_dates=True
)

metadata = pd.read_csv(
    DATA_DIR / "entities.csv"
)

print("Dataset:", returns.shape)
print(
    "Period:",
    returns.index.min(),
    "→",
    returns.index.max()
)

print("Missing:", returns.isna().sum().sum())

Dataset: (1396, 26)
Period: 2021-01-05 00:00:00 → 2026-09-01 00:00:00
Missing: 0


In [3]:
stress_features = pd.DataFrame(
    index=returns.index
)

# Market indicators
stress_features["nifty_return"] = returns["^NSEI"]
stress_features["bank_return"] = returns["^NSEBANK"]

# Macro / external assets
stress_features["oil_return"] = returns["CL=F"]
stress_features["gold_return"] = returns["GC=F"]
stress_features["fx_return"] = returns["INR=X"]

# Entire-system behaviour
stress_features["mean_market_return"] = (
    returns.mean(axis=1)
)

stress_features["market_dispersion"] = (
    returns.std(axis=1)
)

stress_features["worst_asset_return"] = (
    returns.min(axis=1)
)

stress_features["best_asset_return"] = (
    returns.max(axis=1)
)

# Percentage of assets falling
stress_features["negative_breadth"] = (
    (returns < 0).mean(axis=1)
)

# Percentage suffering >2% daily loss
stress_features["severe_loss_breadth"] = (
    (returns < -0.02).mean(axis=1)
)

# Absolute market movement
stress_features["nifty_abs_move"] = (
    returns["^NSEI"].abs()
)

print("Stress features:", stress_features.shape)

stress_features.head()

Stress features: (1396, 12)


,nifty_return,bank_return,oil_return,gold_return,fx_return,mean_market_return,market_dispersion,worst_asset_return,best_asset_return,negative_breadth,severe_loss_breadth,nifty_abs_move
Date,,,,,,,,,,,,
2021-01-05,0.004712,0.016333,0.048509,0.004114,0.003119,0.005441,0.018743,-0.020629,0.063631,0.384615,0.038462,0.004712
2021-01-06,-0.003750,0.002385,0.014020,-0.023455,0.000747,-0.002435,0.015325,-0.028612,0.038390,0.576923,0.115385,0.003750
2021-01-07,-0.000629,0.004972,0.003950,0.002832,-0.000202,0.000699,0.011600,-0.020043,0.025755,0.538462,0.038462,0.000629
2021-01-08,0.014847,0.004012,0.027740,-0.040893,0.001172,0.015530,0.021007,-0.040893,0.059324,0.115385,0.038462,0.014847
2021-01-11,0.009584,-0.002659,0.000191,0.008451,-0.000988,0.013116,0.028492,-0.020226,0.113550,0.307692,0.038462,0.009584


In [4]:
# 5-day market volatility
stress_features["nifty_vol_5d"] = (
    returns["^NSEI"]
    .rolling(5)
    .std()
)

# 20-day market volatility
stress_features["nifty_vol_20d"] = (
    returns["^NSEI"]
    .rolling(20)
    .std()
)

# Banking volatility
stress_features["bank_vol_20d"] = (
    returns["^NSEBANK"]
    .rolling(20)
    .std()
)

# Whole-system average volatility
stress_features["system_vol_20d"] = (
    returns
    .rolling(20)
    .std()
    .mean(axis=1)
)

# Rolling cumulative NIFTY movement
stress_features["nifty_5d_return"] = (
    (1 + returns["^NSEI"])
    .rolling(5)
    .apply(np.prod, raw=True)
    - 1
)

# Rolling cumulative banking movement
stress_features["bank_5d_return"] = (
    (1 + returns["^NSEBANK"])
    .rolling(5)
    .apply(np.prod, raw=True)
    - 1
)

stress_features = stress_features.dropna()

print("Final stress dataset:", stress_features.shape)

stress_features.head()

Final stress dataset: (1377, 18)


,nifty_return,bank_return,oil_return,gold_return,fx_return,mean_market_return,market_dispersion,worst_asset_return,best_asset_return,negative_breadth,severe_loss_breadth,nifty_abs_move,nifty_vol_5d,nifty_vol_20d,bank_vol_20d,system_vol_20d,nifty_5d_return,bank_5d_return
Date,,,,,,,,,,,,,,,,,,
2021-02-02,0.025674,0.035626,0.022596,-0.016283,0.002924,0.027017,0.033468,-0.025252,0.151645,0.115385,0.038462,0.025674,0.029074,0.015869,0.023796,0.024263,0.028721,0.098386
2021-02-03,0.009701,0.014315,0.016983,0.000929,-0.002783,0.008768,0.012905,-0.015450,0.037267,0.230769,0.000000,0.009701,0.025499,0.015954,0.023750,0.024115,0.058883,0.147729
2021-02-04,0.007147,0.016869,0.009697,-0.023633,-0.000799,0.011762,0.024796,-0.034956,0.060900,0.346154,0.076923,0.007147,0.022649,0.015928,0.023891,0.024432,0.078024,0.164255
2021-02-05,0.001920,0.008762,0.011026,0.012298,0.001204,0.004823,0.026680,-0.032666,0.107012,0.384615,0.115385,0.001920,0.018501,0.015910,0.023901,0.024883,0.094587,0.166495
2021-02-08,0.012835,0.009232,0.019701,0.011596,-0.002206,0.016691,0.018849,-0.014496,0.072675,0.192308,0.000000,0.012835,0.008897,0.015836,0.023910,0.024465,0.058440,0.087479


In [5]:
print("===== SYSTEMIC ML DATASET =====")

print("\nShape:")
print(stress_features.shape)

print("\nFeatures:")
for col in stress_features.columns:
    print("-", col)

print("\nMissing values:")
print(stress_features.isna().sum().sum())

print("\nPeriod:")
print(
    stress_features.index.min(),
    "→",
    stress_features.index.max()
)

===== SYSTEMIC ML DATASET =====

Shape:
(1377, 18)

Features:
- nifty_return
- bank_return
- oil_return
- gold_return
- fx_return
- mean_market_return
- market_dispersion
- worst_asset_return
- best_asset_return
- negative_breadth
- severe_loss_breadth
- nifty_abs_move
- nifty_vol_5d
- nifty_vol_20d
- bank_vol_20d
- system_vol_20d
- nifty_5d_return
- bank_5d_return

Missing values:
0

Period:
2021-02-02 00:00:00 → 2026-09-01 00:00:00


In [6]:
# Chronological reference split
split_idx = int(len(stress_features) * 0.70)

reference_data = stress_features.iloc[:split_idx].copy()

print("Reference samples:", len(reference_data))
print("Total samples    :", len(stress_features))

print("\nReference period:")
print(
    reference_data.index.min(),
    "→",
    reference_data.index.max()
)

# Scale features
scaler = StandardScaler()

X_reference = scaler.fit_transform(reference_data)
X_all = scaler.transform(stress_features)

print("\nScaled feature shape:", X_all.shape)

Reference samples: 963
Total samples    : 1377

Reference period:
2021-02-02 00:00:00 → 2024-12-26 00:00:00

Scaled feature shape: (1377, 18)


In [7]:
iso_model = IsolationForest(
    n_estimators=400,
    contamination=0.05,
    max_samples="auto",
    random_state=42,
    n_jobs=-1
)

iso_model.fit(X_reference)

# Higher value = more anomalous
iso_anomaly = -iso_model.score_samples(X_all)

print("Isolation Forest trained.")

print(
    "Anomaly score range:",
    iso_anomaly.min(),
    "to",
    iso_anomaly.max()
)

Isolation Forest trained.
Anomaly score range: 0.35934928883971484 to 0.7247792441337859


In [8]:
pca_model = PCA(
    n_components=0.90,
    random_state=42
)

pca_model.fit(X_reference)

X_pca = pca_model.transform(X_all)

X_reconstructed = pca_model.inverse_transform(
    X_pca
)

pca_error = np.mean(
    (X_all - X_reconstructed) ** 2,
    axis=1
)

print("PCA trained.")

print(
    "Components retained:",
    pca_model.n_components_
)

print(
    "Explained variance:",
    pca_model.explained_variance_ratio_.sum()
)

print(
    "Reconstruction error range:",
    pca_error.min(),
    "to",
    pca_error.max()
)

PCA trained.
Components retained: 8
Explained variance: 0.9033675022131513
Reconstruction error range: 0.006066103538882405 to 1.8700757278740179


In [9]:
svm_model = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

svm_model.fit(X_reference)

# Lower decision function = more abnormal,
# therefore invert it
svm_anomaly = -svm_model.decision_function(
    X_all
)

print("One-Class SVM trained.")

print(
    "Anomaly score range:",
    svm_anomaly.min(),
    "to",
    svm_anomaly.max()
)

One-Class SVM trained.
Anomaly score range: -1.4759980357605809 to 1.6986198226528457


In [10]:
def percentile_score(values):
    """
    Convert anomaly values into percentile scores 0-100.
    Higher = more anomalous / stressed.
    """
    series = pd.Series(values)

    return (
        series.rank(
            method="average",
            pct=True
        ).values * 100
    )


stress_results = stress_features.copy()

stress_results["isolation_score"] = (
    percentile_score(iso_anomaly)
)

stress_results["pca_score"] = (
    percentile_score(pca_error)
)

stress_results["svm_score"] = (
    percentile_score(svm_anomaly)
)

# Ensemble ML stress score
stress_results["ml_stress_score"] = (
    0.40 * stress_results["isolation_score"]
    +
    0.30 * stress_results["pca_score"]
    +
    0.30 * stress_results["svm_score"]
)


def classify_stress(score):

    if score >= 95:
        return "CRITICAL"

    elif score >= 85:
        return "HIGH"

    elif score >= 70:
        return "ELEVATED"

    else:
        return "NORMAL"


stress_results["stress_regime"] = (
    stress_results["ml_stress_score"]
    .apply(classify_stress)
)


print("===== ML SYSTEMIC STRESS ENGINE =====")

print("\nStress score range:")
print(
    stress_results["ml_stress_score"].min(),
    "→",
    stress_results["ml_stress_score"].max()
)

print("\nRegime distribution:")
print(
    stress_results[
        "stress_regime"
    ].value_counts()
)

print("\nTop 10 most stressed market days:")

display(
    stress_results[
        [
            "ml_stress_score",
            "stress_regime",
            "nifty_return",
            "bank_return",
            "market_dispersion",
            "negative_breadth",
            "severe_loss_breadth"
        ]
    ]
    .sort_values(
        "ml_stress_score",
        ascending=False
    )
    .head(10)
)

===== ML SYSTEMIC STRESS ENGINE =====

Stress score range:
1.6848220769789395 → 99.95642701525054

Regime distribution:
stress_regime
NORMAL      1068
ELEVATED     177
HIGH         101
CRITICAL      31
Name: count, dtype: int64

Top 10 most stressed market days:


,ml_stress_score,stress_regime,nifty_return,bank_return,market_dispersion,negative_breadth,severe_loss_breadth
Date,,,,,,,
2026-04-08,99.956427,CRITICAL,0.037784,0.056674,0.048547,0.192308,0.038462
2024-06-04,99.382716,CRITICAL,-0.059294,-0.079469,0.051884,0.923077,0.576923
2022-03-02,98.620189,CRITICAL,-0.011192,-0.022994,0.038169,0.692308,0.269231
2025-05-12,98.533043,CRITICAL,0.038183,0.033354,0.028229,0.115385,0.076923
2022-03-07,98.533043,CRITICAL,-0.023527,-0.044657,0.042894,0.769231,0.576923
2021-02-02,98.322440,CRITICAL,0.025674,0.035626,0.033468,0.115385,0.038462
2022-02-24,98.082789,CRITICAL,-0.047781,-0.057872,0.025651,0.884615,0.884615
2024-06-05,97.988381,CRITICAL,0.033624,0.045303,0.014699,0.000000,0.000000
2021-02-26,97.959332,CRITICAL,-0.037636,-0.047755,0.018955,0.961538,0.769231


In [11]:
stress_results["iso_alert"] = (
    stress_results["isolation_score"] >= 95
)

stress_results["pca_alert"] = (
    stress_results["pca_score"] >= 95
)

stress_results["svm_alert"] = (
    stress_results["svm_score"] >= 95
)

stress_results["model_agreement"] = (
    stress_results[
        ["iso_alert", "pca_alert", "svm_alert"]
    ]
    .sum(axis=1)
)

agreement_table = (
    stress_results[
        [
            "ml_stress_score",
            "stress_regime",
            "isolation_score",
            "pca_score",
            "svm_score",
            "model_agreement"
        ]
    ]
    .sort_values(
        "ml_stress_score",
        ascending=False
    )
    .head(15)
)

agreement_table

,ml_stress_score,stress_regime,isolation_score,pca_score,svm_score,model_agreement
Date,,,,,,
2026-04-08,99.956427,CRITICAL,100.000000,99.854757,100.000000,3
2024-06-04,99.382716,CRITICAL,99.927378,100.000000,98.039216,3
2022-03-02,98.620189,CRITICAL,99.273784,98.620189,97.748729,3
2025-05-12,98.533043,CRITICAL,98.838054,97.240378,99.419027,3
2022-03-07,98.533043,CRITICAL,99.709513,97.966594,97.530864,3
2021-02-02,98.322440,CRITICAL,99.564270,98.547567,96.441540,3
2022-02-24,98.082789,CRITICAL,99.782135,98.257081,95.642702,3
2024-06-05,97.988381,CRITICAL,99.055919,98.329702,96.223675,3
2021-02-26,97.959332,CRITICAL,99.854757,96.877269,96.514161,3


In [12]:
critical_days = stress_results[
    stress_results["stress_regime"] == "CRITICAL"
]

high_confidence = critical_days[
    critical_days["model_agreement"] >= 2
]

print("===== ML ENGINE QUALITY CHECK =====")

print(
    "Total days:",
    len(stress_results)
)

print(
    "Critical days:",
    len(critical_days)
)

print(
    "Critical days with >=2 model agreement:",
    len(high_confidence)
)

print(
    "Critical days with all 3 models agreeing:",
    (critical_days["model_agreement"] == 3).sum()
)

print("\nAverage stress score:")
print(
    stress_results["ml_stress_score"].mean()
)

print("\nTop high-confidence stress days:")

display(
    high_confidence[
        [
            "ml_stress_score",
            "stress_regime",
            "model_agreement",
            "nifty_return",
            "bank_return",
            "negative_breadth",
            "severe_loss_breadth"
        ]
    ]
    .sort_values(
        "ml_stress_score",
        ascending=False
    )
    .head(10)
)

===== ML ENGINE QUALITY CHECK =====
Total days: 1377
Critical days: 31
Critical days with >=2 model agreement: 29
Critical days with all 3 models agreeing: 10

Average stress score:
50.03631082062454

Top high-confidence stress days:


,ml_stress_score,stress_regime,model_agreement,nifty_return,bank_return,negative_breadth,severe_loss_breadth
Date,,,,,,,
2026-04-08,99.956427,CRITICAL,3,0.037784,0.056674,0.192308,0.038462
2024-06-04,99.382716,CRITICAL,3,-0.059294,-0.079469,0.923077,0.576923
2022-03-02,98.620189,CRITICAL,3,-0.011192,-0.022994,0.692308,0.269231
2025-05-12,98.533043,CRITICAL,3,0.038183,0.033354,0.115385,0.076923
2022-03-07,98.533043,CRITICAL,3,-0.023527,-0.044657,0.769231,0.576923
2021-02-02,98.322440,CRITICAL,3,0.025674,0.035626,0.115385,0.038462
2022-02-24,98.082789,CRITICAL,3,-0.047781,-0.057872,0.884615,0.884615
2024-06-05,97.988381,CRITICAL,3,0.033624,0.045303,0.000000,0.000000
2021-02-26,97.959332,CRITICAL,3,-0.037636,-0.047755,0.961538,0.769231


In [13]:
downside_raw = (
    0.35 * stress_results["negative_breadth"]
    +
    0.35 * stress_results["severe_loss_breadth"]
    +
    0.15 * (stress_results["nifty_return"] < 0).astype(float)
    +
    0.15 * (stress_results["bank_return"] < 0).astype(float)
)

stress_results["downside_stress"] = (
    downside_raw * 100
)

stress_results["ml_market_risk_score"] = (
    0.70 * stress_results["ml_stress_score"]
    +
    0.30 * stress_results["downside_stress"]
).clip(0, 100)

stress_results[
    [
        "ml_stress_score",
        "downside_stress",
        "ml_market_risk_score"
    ]
].describe()

,ml_stress_score,downside_stress,ml_market_risk_score
count,1377.000000,1377.000000,1377.000000
mean,50.036311,33.460142,45.063460
std,23.553873,23.132090,18.861577
min,1.684822,0.000000,6.429375
25%,31.684822,10.769231,30.423077
50%,48.518519,32.500000,42.957684
75%,66.978940,55.576923,58.502165
max,99.956427,91.923077,96.234875


In [14]:
import joblib

joblib.dump(
    scaler,
    MODEL_DIR / "stress_scaler.pkl"
)

joblib.dump(
    iso_model,
    MODEL_DIR / "isolation_forest.pkl"
)

joblib.dump(
    pca_model,
    MODEL_DIR / "stress_pca.pkl"
)

joblib.dump(
    svm_model,
    MODEL_DIR / "oneclass_svm.pkl"
)

stress_results.to_csv(
    DATA_DIR / "ml_stress_scores.csv"
)

print("Saved:")
print("- stress_scaler.pkl")
print("- isolation_forest.pkl")
print("- stress_pca.pkl")
print("- oneclass_svm.pkl")
print("- ml_stress_scores.csv")

Saved:
- stress_scaler.pkl
- isolation_forest.pkl
- stress_pca.pkl
- oneclass_svm.pkl
- ml_stress_scores.csv


In [15]:
print("===== FINCASCADE ML ENGINE COMPLETE =====")

print("\nModels:")
print("1. Isolation Forest")
print("2. PCA Reconstruction")
print("3. One-Class SVM")

print("\nInput features:")
print(len(stress_features.columns))

print("\nHistorical observations:")
print(len(stress_results))

print("\nStress regimes:")
print(
    stress_results["stress_regime"].value_counts()
)

print("\nCritical days:")
print(
    (stress_results["stress_regime"] == "CRITICAL").sum()
)

print("\nCritical days with >=2 model agreement:")
print(
    (
        (stress_results["stress_regime"] == "CRITICAL")
        &
        (stress_results["model_agreement"] >= 2)
    ).sum()
)

print("\nMaximum ML Stress Score:")
print(
    stress_results["ml_stress_score"].max()
)

print("\nMaximum downside-aware ML Market Risk Score:")
print(
    stress_results["ml_market_risk_score"].max()
)

print("\nSaved models:")

for file in [
    "stress_scaler.pkl",
    "isolation_forest.pkl",
    "stress_pca.pkl",
    "oneclass_svm.pkl"
]:
    print(
        file,
        "→",
        (MODEL_DIR / file).exists()
    )

print("\nML STRESS ENGINE COMPLETE ✅")

===== FINCASCADE ML ENGINE COMPLETE =====

Models:
1. Isolation Forest
2. PCA Reconstruction
3. One-Class SVM

Input features:
18

Historical observations:
1377

Stress regimes:
stress_regime
NORMAL      1068
ELEVATED     177
HIGH         101
CRITICAL      31
Name: count, dtype: int64

Critical days:
31

Critical days with >=2 model agreement:
29

Maximum ML Stress Score:
99.95642701525054

Maximum downside-aware ML Market Risk Score:
96.23487514663985

Saved models:
stress_scaler.pkl → True
isolation_forest.pkl → True
stress_pca.pkl → True
oneclass_svm.pkl → True

ML STRESS ENGINE COMPLETE ✅
